# Measure objects and crop images

**Purpose.** Compute per-object morphology, intensity, and spatial measurements from merged image-mask arrays, with optional object-centred crops.

**Recommended use.** Use after segmentation when quantitative features or classifier training images are required.

**Primary outputs.** `measurements/measurements.db` and, when enabled, object crops under `data/`.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.measure.measure_crop`](https://einarolafsson.github.io/spacr/api/spacr/measure/index.html#spacr.measure.measure_crop)

```python
measure_crop(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.measure import measure_crop

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.measure.measure_crop`](https://einarolafsson.github.io/spacr/api/spacr/measure/index.html#spacr.measure.measure_crop)

> Organelle parameters are placed in a separate code cell to keep the primary segmentation configuration readable.


#### Input & Experiment

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`experiment`** *(optional)* — (str) - Free-text run label. Its real effect is naming the exported PNG dataset tar as &lt;YYMMDD&gt;_&lt;experiment&gt;.tar (a random-numbered variant is used if that name already exists), so give each screen a distinct value to avoid confusing dataset tars. It is also passed to the measurement-database writer but not stored there. Defaults vary by pipeline: 'exp', 'exp.' or 'experiment_1'.

#### Mask & Channel Mapping

- **`channels`** *(optional)* — (list of int) - Zero-indexed image channels kept in merged/*.npy and measured by measure_crop; each entry produces its own &lt;object&gt;_channel_&lt;n&gt;_* intensity columns. The list length fixes where masks land, so cell/nucleus/pathogen_mask_dim must shift if you change it. Preprocessing silently resets it to range(n) when it does not match the number of channel folders found. Default [0,1,2,3]. External Masks starts with []; there an empty list means every detected intensity channel, not no channels.
- **`cell_mask_dim`** *(optional)* — (int) - Position along the last axis of each merged/*.npy array where the cell label mask sits. Merged arrays are ordered [image channels..., cell, nucleus, pathogen, organelle], so the default 4 assumes the four channels 0-3 were kept; keep fewer channels and every mask dim shifts down. None makes measure_crop skip all cell measurements and cell crops. Default 4.
- **`nucleus_mask_dim`** *(optional)* — (int) - Position along the last axis of each merged/*.npy array where the nucleus label mask sits, one plane after the cell mask. With the default four image channels (0-3) that is 5; keep a different number of channels and it shifts by the same amount. None makes measure_crop skip nucleus measurements and cell-to-nucleus linking. Default 5.
- **`pathogen_mask_dim`** *(optional)* — (int) - Position along the last axis of each merged/*.npy array where the pathogen label mask sits, one plane after the nucleus mask. With the default four image channels (0-3) that is 6; shift it if you keep a different number of channels. None makes measure_crop skip pathogen measurements, so infection status cannot be scored. Default 6.
- **`cytoplasm`** *(optional)* — (bool) - Derive a cytoplasm object per cell (cell mask with nucleus, pathogen and organelle pixels removed) and write it to its own cytoplasm table, which recruitment ratios such as pathogen/cytoplasm intensity are computed from. Requires a cell mask; measure_crop switches it on automatically whenever cell_mask_dim is set, so the value you enter is usually overridden. Default False.
- **`timelapse`** *(optional)* — (bool) - Treat each well/field as a time series instead of independent images: files are grouped into time stacks, randomization is switched off, per-channel movies are written, objects in timelapse_objects are tracked across frames, a timeID column is added to the measurement tables, and measure_crop stops writing single-object PNGs. Only enable when filenames carry a time index. Default False.
- **`timelapse_objects`** *(optional)* — (list) - Which segmented objects are tracked across frames and relabelled with track IDs: any subset of ['cell', 'nucleus', 'pathogen']; any other value aborts the run with a message. Each extra entry costs a full additional tracking pass. Tracking nuclei is often more stable than cells when cells touch. Default ['cell'].

#### Illumination Correction

- **`illumination_correction`** *(optional)* — (bool) - Estimate the uneven illumination of the microscope from the fields themselves and divide it out before any intensity feature is measured. Off by default. On, the same cell measures the same wherever it sits in the field of view, which is what removes the position-dependent bias behind plate edge effects. Default False.
- **`illumination_model`** *(optional)* — (str) - Path to an illumination model saved earlier. Empty means estimate a fresh one from the fields in src, which is what you want unless you are re-measuring a plate and must reproduce the exact correction an earlier run applied. Default empty.
- **`illumination_estimator`** *(optional)* — (str) - How the smooth field is fitted to the across-field median: 'polynomial' fits a low-order surface, which cannot bend around a cell and is the right choice for a lamp profile plus a vignette; 'smooth' Gaussian-blurs the median instead and can follow a dust shadow the polynomial would miss. Default polynomial.
- **`illumination_degree`** *(optional)* — (int) - Order of the fitted illumination surface. 4 gives fifteen terms, enough for a lamp profile, a vignette and a tilt. Raising it lets the surface follow finer structure and, past about 6, start absorbing the cells you are trying to measure. Default 4.
- **`illumination_dark`** *(optional)* — (float) - Camera dark offset in raw counts, subtracted before the gain is applied. Leave at zero unless you measured it from a dark frame: it is not identifiable from the images themselves, and an estimated value can subtract genuine background signal. Default 0.0.
- **`illumination_per_plate`** *(optional)* — (bool) - Estimate one illumination field per plate rather than one for every plate together. Lamp age, a re-seated filter cube or a different objective change the field between acquisition sessions, so pooling two sessions estimates neither of them well. Default True.
- **`illumination_max_fields`** *(optional)* — (int) - How many fields per plate the estimate reads, sampled evenly across the plate. More fields make the across-field median a better object rejector and cost linear time; below about ten, cells start surviving into the gain map. Default 50.
- **`illumination_qc`** *(optional)* — (bool) - Write the QC figure beside the model: the estimated field as an image, the intensity-versus-position trend before and after, and the percentage of the position bias the correction removed. The figure has low computational cost and provides direct verification of the correction. Default True.
- **`illumination_on_missing`** *(optional)* — (str) - What to do with a field whose plate the model does not cover: 'error' fails that field and stamps the run incomplete, 'skip' measures it uncorrected. Default error, because corrected and uncorrected rows sharing one table is worse than a failed field.

#### Measurement Features

- **`save_measurements`** *(optional)* — (bool) - Master switch for the measurement half of measure_crop: compute morphology and intensity features for every cell, nucleus, pathogen, organelle and cytoplasm object and write them to the plate's SQLite database. Set it False when you only want cropped PNGs or filtered masks -- segmentation and cropping still run, but no measurement tables are written. Default True.
- **`calculate_correlation`** *(optional)* — (bool) - For every pair of measured channels and every object mask, compute a per-object Pearson correlation plus Manders M1/M2 at each cut-off in manders_thresholds, stored as &lt;object&gt;_channel_i_channel_j_* columns. Needs at least two channels. Turn it off to cut measurement time and database size when colocalisation is not part of the phenotype. Default True.
- **`corrected_manders`** *(optional)* — (bool) - Add standards-compliant Manders coefficients (manders_m1, manders_m2 and manders_overlap_coefficient) alongside the deprecated M1_correlation_* columns, which use a different definition. Existing columns remain unchanged for compatibility across measurement runs. Use the new columns for Manders analyses. Default False.
- **`spatial_measurements`** *(optional)* — (bool) - Measure each object's neighbourhood: the number of neighbours within a radius, first and second nearest-neighbour distances, and the fraction of its border contacting another object. These measurements can be used to model density-associated variation in morphology and intensity. They are not produced for cytoplasm, which is defined as one object per cell. Computation requires one KD-tree and one boundary pass per field. Default False.
- **`spatial_neighbor_radius`** *(optional)* — (int) - Radius used by spatial_measurements when counting neighbouring objects. The value is expressed in the units recorded for the measurement table: pixels for two-dimensional data and micrometres for calibrated three-dimensional data. The radius is included in the output column name, so use one value consistently across plates that will be combined. Ignored unless spatial_measurements is enabled. Default 50.
- **`manders_thresholds`** *(optional)* — (list) - Percentiles (0-100) at which Manders' overlap coefficients are computed. For each object, each entry thresholds both channels at that percentile; pixels above both count as overlap, and M1/M2 report each channel's fraction of total object intensity there, saved as M1_correlation_&lt;t&gt; and M2_correlation_&lt;t&gt;. High values isolate the brightest puncta. Requires calculate_correlation. Default [15, 85, 95].
- **`homogeneity`** *(optional)* — (bool) - Compute grey-level co-occurrence-matrix homogeneity for every object in every channel, adding one homogeneity_distance_&lt;d&gt; column per entry in homogeneity_distances. Homogeneity is high for smooth, evenly filled objects and low for punctate or grainy ones, so keep it on for texture phenotypes; disabling it noticeably speeds up measurement. Default True.
- **`homogeneity_distances`** *(optional)* — (list) - Pixel offsets used to build each object's grey-level co-occurrence matrix; every entry adds one homogeneity_distance_&lt;d&gt; feature per channel. Small offsets capture fine-grained texture, large ones capture coarse structure, and offsets larger than the object itself carry no signal. More entries means more features and slower measurement. Default [8, 16, 32].
- **`radial_dist`** *(optional)* — (bool) - Measure how each channel's intensity varies with distance from the nucleus, pathogen and organelle boundaries inside each cell, binned into 6 shells and saved as &lt;object&gt;_rad_dist_channel_&lt;c&gt;_bin_0-5. Keep it on to quantify recruitment or intensity gradients toward an object; turn it off to shrink the feature table and speed up measurement. Default True.
- **`distance_gaussian_sigma`** *(optional)* — (int or None) - Sigma in pixels of the Gaussian blur applied to each channel before measuring intensity-weighted centroid distances from cells to nuclei and pathogens. Larger values smooth out speckle so the weighted centroid follows broad signal. None or 0 skips these distance features entirely. Needs a cell mask plus a nucleus or pathogen mask. Default 10.
- **`object_distances`** *(optional)* — (bool) - Measure every distance between and within objects: centre to centre, centre to the nearest surface of each other object type (both directions), surface to surface -- which is zero when two objects touch and is what 'how far apart are they' means -- the overlap fraction, how far the centre sits from its own boundary, and how close the object is to the edge of the field. Off by default because the calculation is computationally expensive on a 3-D field. The cost is one distance transform per object type per field, not one per pair of objects.
- **`object_distance_maxima`** *(optional)* — (bool) - Also find the intensity maxima inside each object and measure where they are: how many, how spread out, and how far each is from the object's own boundary, its centre, and the nearest surface of every other object type. The most expensive part of object_distances, and ignored when that is off. Default True.
- **`object_distance_intensity`** *(optional)* — (bool) - Include the families that need the intensity images: the local maxima above and the scalar displacement between each channel's intensity-weighted centre of mass and the geometric centroid. False measures geometry only. Ignored when object_distances is off. Default True.

#### Object Filtering

- **`uninfected`** *(optional)* — (bool) - Decides which cells survive the consistency filter in measure_crop. True keeps any cell that has both a nucleus and a cytoplasm; False also demands at least one pathogen, dropping uninfected cells from every table. Either way, nucleus/pathogen/cytoplasm labels outside the surviving cells are zeroed. Only applied when cell, nucleus and pathogen masks all exist; forced True otherwise. Default True.
- **`cell_min_size`** *(optional)* — (int) - (Deprecated) Pixel-area floor applied to cell labels during measurement: any cell smaller than this is erased from the mask before features are extracted. Superseded by cell_min_area, which filters at segmentation time, but this one still runs if you set it. 0 or None disables it. Default 0.
- **`cell_max_size`** *(optional)* — (int | None) - Drop cells larger than this many pixels. None, the default, disables the filter and preserves prior behavior. The minimum sizes remove debris; only a maximum removes a segmentation artifact, which passes every minimum and carries its area into the classifier and the regression. The run prints how many objects each bound dropped.
- **`cytoplasm_min_size`** *(optional)* — (int) - (Deprecated) Pixel-area floor for the cytoplasm mask, which is the cell mask with nucleus, pathogen and organelle pixels removed. Cytoplasm regions below this are erased before measurement, so their host cell yields no cytoplasm features and any recruitment ratio built on them is lost. 0 or None disables. Default 0.
- **`nucleus_min_size`** *(optional)* — (int) - (Deprecated) Minimum nucleus size in pixels^2 applied during measure_crop: labels covering fewer pixels than this are erased from the nucleus mask before any feature is measured, so those nuclei never reach the database. 0 (default) disables it. Prefer nucleus_min_area, which filters at segmentation time.
- **`nucleus_max_size`** *(optional)* — (int | None) - Drop nuclei larger than this many pixels, counted the same way nucleus_min_size counts them. None -- the default -- disables it and preserves prior behavior. A minimum removes debris; only a maximum removes nuclei merged into a single mask, which pass every minimum and then bias downstream area and DNA-content measurements. The run prints how many objects each bound dropped.
- **`pathogen_min_size`** *(optional)* — (int) - (Deprecated) Minimum pathogen object area in pixels squared, applied during measurement: any label with fewer pixels than this is erased from the pathogen mask before features are extracted. 0, the default, disables it. Superseded by pathogen_min_area, which filters at segmentation time instead.
- **`pathogen_max_size`** *(optional)* — (int | None) - Drop pathogens larger than this many pixels, counted the same way pathogen_min_size counts them. None -- the default -- disables it and preserves prior behavior. This bound removes a vacuole of tightly packed parasites segmented as one object: it passes every minimum size and inflates both the per-cell burden and the mean parasite area. The run prints how many objects each bound dropped.
- **`merge_edge_pathogen_cells`** *(optional)* — (bool) - During measurement, reconcile pathogens straddling two host-cell masks: if 90 percent or more of the pathogen lies in one cell, its pixels in the neighbours are erased; otherwise the overlapping cell labels are fused into a single cell. Switch off to keep the raw cell segmentation when parasites legitimately touch two cells. Default True.

#### Crop Output

- **`save_png`** *(optional)* — (bool) - Write one PNG crop per segmented object into &lt;crop_mode&gt;_png/ and register each path in the png_list table of measurements.db. Required for training or applying a classifier, for the Annotate app and for the UMAP image plots. Turn off to only compute measurements and save time and disk. Default True.
- **`save_arrays`** *(optional)* — (bool) - Also save each object as a raw .npy array - all channels, cropped to its bounding box, unnormalised - under a region_array/ folder. Enable when you need full bit depth or channels beyond png_dims for custom analysis; it uses far more disk than PNGs. Requires save_png to be True as well. Default False.
- **`crop_mode`** *(required)* — (list) - Mask used to center each PNG crop: 'cell', 'nucleus', 'pathogen', 'cytoplasm' or 'organelle'. One crop set is written per entry into &lt;mode&gt;_png/ folders, so ['cell','nucleus'] doubles the images written and the rows added to png_list. A single png_size such as [224,224] is broadcast to every mode, as are dialate_pngs and dialate_png_ratios; use lists only when modes require different values. A list shorter than crop_mode reuses its final entry for the remaining modes and records a warning. Default ['cell'].
- **`png_size`** *(optional)* — (list of int) - Output crop size as [width, height] in pixels, centred on the object centroid; larger keeps more surroundings, smaller clips large objects. Should match the classifier input size (default [224,224]). With several crop_mode entries pass a list of lists, one size per mode, or a single size is reused for all.
- **`png_channel_mapping`** *(optional)* — (dict) - Which source channel goes in each colour of the saved PNG, e.g. {'r': 2, 'g': 1, 'b': 0}: channel 2 is red, 1 is green, 0 is blue. Says outright what png_dims only implied. Channels not named are absent from the crops (measurements are unaffected); a colour left blank is an empty plane. Naming the same channel for all three writes a greyscale PNG. Default {'r': 2, 'g': 1, 'b': 0}, which for a standard 405/488/555 stack puts the nuclear stain in blue.
- **`dialate_pngs`** *(optional)* — (bool) - Grow each object mask before cropping so the PNG keeps a rim of surrounding pixels instead of a hard mask edge; the amount comes from dialate_png_ratios. May be a list with one value per crop_mode entry (a single value applies to all of them), and is forced off for crop_mode 'cytoplasm'. Enable when context around the object helps the classifier. Default False.
- **`dialate_png_ratios`** *(optional)* — (list of float) - Dilation amount as a fraction of object size: the mask is grown by ratio * sqrt(object area) pixels of binary dilation, so 0.2 expands a cell by roughly 20% of its diameter and pulls in surrounding background. Only used when dialate_pngs is True. A single value applies to every crop_mode entry; pass a list only when the modes need different ratios. Default [0.2].
- **`use_bounding_box`** *(optional)* — (bool) - Crop the object's rectangular bounding box padded by 10 px instead of its mask, so neighbouring cells and background inside the box are kept rather than zeroed out. Enable when the classifier should see local context; leave off to isolate a single object on a black background. Default False.
- **`normalize`** *(conditionally required)* — (bool or list) - Control percentile normalization before display, model input, or crop export. Display and activation-map tools use True for a 2nd-to-98th-percentile stretch. Measure and External Masks start at False; Measure accepts False or a two-number [low, high] percentile pair and refuses bare True because it supplies no bounds. It affects display and exported-crop scaling, not measured source intensities. Default True in the display-oriented tools.
- **`normalize_by`** *(optional)* — (str) - Percentile source used to rescale cropped PNGs, and only active when 'normalize' is a [low, high] percentile pair: 'png' stretches each crop to its own percentiles, maximising per-object contrast; 'fov' uses percentiles from the whole field, keeping brightness comparable between objects. Choose 'fov' if crop intensities will be compared. Default 'png'.

#### Preview & Diagnostics

- **`plot`** *(optional)* — (bool) - Render and save quality-control figures during the pipeline, including channel montages, Cellpose mask overlays, filtration comparisons, and crop grids. Figure generation increases runtime and memory use, particularly for complete plates. test_mode enables this setting automatically. Default False. Merged Classifier and Recruitment both start with plotting enabled so their diagnostic figures are produced on the first run.
- **`test_mode`** *(optional)* — (bool) - Run the pipeline on a small random subset instead of the whole folder. Mask generation copies test_images (default 10) complete image sets into &lt;src&gt;/test and works there; measure_crop copies test_nr (default 10) merged arrays into test/merged. Both also force verbose and plot on. Use it to check channel assignment, diameters and thresholds before committing to a full plate. Default False.
- **`test_nr`** *(optional)* — (int) - How many files are sampled at random from merged/ into test/merged when test_mode is on in the measure-and-crop pipeline, so measurement runs on a small subset. Raise it if a handful of fields is not representative; each extra file costs a full measurement pass. Default 10.

#### 3D Calibration (Beta)

- **`anisotropy`** *(optional)* — (float or None) - Ratio of z step to xy pixel size (dz / dxy), used by volumetric mode to represent inter-plane distance. A value of 1.0 on a confocal stack whose z step is 3-10 times the xy pixel size can fuse objects along z. Leave this None and set voxel_size_z_um / voxel_size_xy_um to derive it; if neither is available, volumetric mode raises an error rather than assuming 1.0. Measure also uses it for 3-D region properties and distance transforms. Default None.
- **`voxel_size_z_um`** *(optional)* — (float or None) - Spacing between consecutive z planes in micrometres, obtained from the acquisition metadata. Together with voxel_size_xy_um it determines anisotropy and converts object volumes from voxel counts into cubic micrometres. Changing it rescales every physical z quantity and the anisotropy used for segmentation; it has no effect on a 'project' run. Measure uses the pair to report 3-D morphology in physical units and records the units in measurement_units. Default None.
- **`voxel_size_xy_um`** *(optional)* — (float or None) - Width of one pixel in micrometres in the image plane, assumed square. Used with voxel_size_z_um to derive anisotropy and to turn voxel counts into physical volumes and surface areas. Note this is a different setting from um_per_pixel, which only sizes the scale bar drawn on figures and never reaches a measurement. This one does reach measurements, but only on a 3-D run: a 2-D run never applies it, because doing so would turn every *_area from px2 into um2 under an unchanged column name. Default None.

#### Runtime & Reliability

- **`resume`** *(optional)* — (bool) - Continue an interrupted run from its last validated boundary. Mask revalidates existing mask and merged arrays; Measure accepts only fields complete in every owned table and clears partial rows before retrying; and Format Converter reopens each checkpointed TIFF. These validations reduce the risk of reusing partial output and require additional reads during resumption. Default False.
- **`strict_errors`** *(optional)* — (bool or None) - Error-handling policy for recoverable steps. Off records failures in the run ledger and final summary while continuing with successful items. On raises immediately for setup or configuration errors such as unreadable paths, missing columns or inaccessible databases, preventing partial batch results from invalid inputs. Per-item failures such as one corrupt image remain recoverable under either policy. None defers to $SPACR_STRICT_ERRORS. Default None.
- **`max_failure_rate`** *(optional)* — (float or None) - Fraction of failed items above which the run aborts. For example, 0.2 aborts after more than 20% of items fail. The failure ledger is written to the artifact before the abort. None disables rate-based abortion; failures remain counted and reported, and incomplete artifacts are marked partial. Default None.
- **`dry_run`** *(optional)* — (bool) - Validate settings against the selected data, report the planned operations, and stop before any compute begins. Checks that src contains the expected files, channel and mask-plane indices are valid, and required models, barcode CSV files, or measurements.db files are present. Each problem is reported with a suggested correction, followed by a summary of the planned segmentation, measurement, and output locations. Nothing is written and no model is loaded. Default False.
- **`verbose`** *(optional)* — (bool) - Print the resolved settings table, channel and model choices per object type, row counts per table, and object counts after each filter. It only adds console output; enable it to identify which stage produced an unexpected object count. The default is True for mask, UMAP, screen analysis, barcode mapping and Cellpose training, and False for measure, plotting helpers and regression. Invasion and Replication also start with console detail disabled.
- **`n_jobs`** *(optional)* — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.

#### Mask & Channel Mapping

- **`number_of_organelles`** *(optional)* — (int) - How many organelle slots this run has, from 0 to 26. Each slot is an independent object with its own channel, its own type preset and its own copy of every detection setting, named organelle_*, organelleb_*, organellec_* and so on; raising the number generates another slot's settings and lowering it hides the slots above the new number without deleting them. A hidden slot keeps its values, is still written to the settings file, and comes back exactly as it was when the number is raised again, so a smaller number can be tried without losing work. Default 0.
- **`organelle_mask_dim`** *(optional)* — (int) - Position along the last axis of each merged/*.npy array where the organelle label mask sits. Masks follow the image channels in the order cell, nucleus, pathogen, organelle, so with four channels and all three other masks present it is 7. Leave it unset/None and organelles are not measured at all. No default is applied.
- **`organelleb_mask_dim`** *(optional)* — (int) - Position along the last axis of each merged/*.npy array where the organelle 2 label mask sits. Masks follow the image channels in the order cell, nucleus, pathogen, organelle, so with four channels and all three other masks present it is 7. Leave it unset/None and organelles are not measured at all. No default is applied.
- **`organellec_mask_dim`** *(optional)* — (int) - Position along the last axis of each merged/*.npy array where the organelle 3 label mask sits. Masks follow the image channels in the order cell, nucleus, pathogen, organelle, so with four channels and all three other masks present it is 7. Leave it unset/None and organelles are not measured at all. No default is applied.
- **`organelled_mask_dim`** *(optional)* — (int) - Position along the last axis of each merged/*.npy array where the organelle 4 label mask sits. Masks follow the image channels in the order cell, nucleus, pathogen, organelle, so with four channels and all three other masks present it is 7. Leave it unset/None and organelles are not measured at all. No default is applied.
- **`organelle_type`** *(optional)* — (str) - Organelle morphology used to populate recommended detection settings; explicitly configured values are not overwritten. Options are 'punctate', 'vesicular', 'spherical', 'filamentous', 'tubular', 'reticular', 'cisternal', 'toroidal' and 'crescent'. Morphology alone does not determine the detector: 'vesicular' and 'spherical' also use organelle_diameter because a 200 nm vesicle appears punctate whereas a 2 µm vacuole appears annular. Default 'custom', which applies no recommendations.
- **`organelleb_type`** *(optional)* — (str) - Organelle morphology used to populate recommended detection settings; explicitly configured values are not overwritten. Options are 'punctate', 'vesicular', 'spherical', 'filamentous', 'tubular', 'reticular', 'cisternal', 'toroidal' and 'crescent'. Morphology alone does not determine the detector: 'vesicular' and 'spherical' also use organelleb_diameter because a 200 nm vesicle appears punctate whereas a 2 µm vacuole appears annular. Default 'custom', which applies no recommendations.
- **`organellec_type`** *(optional)* — (str) - Organelle morphology used to populate recommended detection settings; explicitly configured values are not overwritten. Options are 'punctate', 'vesicular', 'spherical', 'filamentous', 'tubular', 'reticular', 'cisternal', 'toroidal' and 'crescent'. Morphology alone does not determine the detector: 'vesicular' and 'spherical' also use organellec_diameter because a 200 nm vesicle appears punctate whereas a 2 µm vacuole appears annular. Default 'custom', which applies no recommendations.
- **`organelled_type`** *(optional)* — (str) - Organelle morphology used to populate recommended detection settings; explicitly configured values are not overwritten. Options are 'punctate', 'vesicular', 'spherical', 'filamentous', 'tubular', 'reticular', 'cisternal', 'toroidal' and 'crescent'. Morphology alone does not determine the detector: 'vesicular' and 'spherical' also use organelled_diameter because a 200 nm vesicle appears punctate whereas a 2 µm vacuole appears annular. Default 'custom', which applies no recommendations.

#### Measurement Features

- **`summarize_organelles_by`** *(optional)* — (str, list or None) - Parent compartments to roll every enabled organelle slot into. Accepts 'cell', 'nucleus', 'pathogen' and 'cytoplasm'; each writes one &lt;parent&gt;_organelle_summary row per parent with a separate organelle_summary_&lt;slot&gt;_* column family. Raw per-organelle tables are always written when their mask dim is enabled. Default 'cell'; None disables only these rollups.

#### Object Filtering

- **`organelle_min_size`** *(optional)* — (int) - Objects below this area are removed from organelle masks, so raise it to clear dim specks and hot pixels or lower it to keep faint puncta. Set square pixels; default 10, and 0 disables it. Classical segmenters and the U-Net apply it during segmentation (LoG/DoG do not; the ring method uses one quarter, minimum 3), then the final label image applies it again. This deprecated setting remains active. Measure and External Masks start every organelle slot at 0 because they consume existing labels instead of segmenting new ones.
- **`organelleb_min_size`** *(optional)* — (int) - Objects below this area are removed from organelle 2 masks, so raise it to clear dim specks and hot pixels or lower it to keep faint puncta. Set square pixels; default 10, and 0 disables it. Classical segmenters and the U-Net apply it during segmentation (LoG/DoG do not; the ring method uses one quarter, minimum 3), then the final label image applies it again. This deprecated setting remains active. Measure and External Masks start every organelle 2 slot at 0 because they consume existing labels instead of segmenting new ones.
- **`organellec_min_size`** *(optional)* — (int) - Objects below this area are removed from organelle 3 masks, so raise it to clear dim specks and hot pixels or lower it to keep faint puncta. Set square pixels; default 10, and 0 disables it. Classical segmenters and the U-Net apply it during segmentation (LoG/DoG do not; the ring method uses one quarter, minimum 3), then the final label image applies it again. This deprecated setting remains active. Measure and External Masks start every organelle 3 slot at 0 because they consume existing labels instead of segmenting new ones.
- **`organelled_min_size`** *(optional)* — (int) - Objects below this area are removed from organelle 4 masks, so raise it to clear dim specks and hot pixels or lower it to keep faint puncta. Set square pixels; default 10, and 0 disables it. Classical segmenters and the U-Net apply it during segmentation (LoG/DoG do not; the ring method uses one quarter, minimum 3), then the final label image applies it again. This deprecated setting remains active. Measure and External Masks start every organelle 4 slot at 0 because they consume existing labels instead of segmenting new ones.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Input & Experiment
    # Required settings
    'src': 'path',
    # Optional settings
    'experiment': 'exp',

    # Mask & Channel Mapping
    # Optional settings
    'channels': [0, 1, 2, 3],
    'cell_mask_dim': 4,
    'nucleus_mask_dim': 5,
    'pathogen_mask_dim': 6,
    'cytoplasm': False,
    'timelapse': False,
    'timelapse_objects': ['cell'],

    # Illumination Correction
    # Optional settings
    'illumination_correction': False,
    'illumination_model': '',
    'illumination_estimator': 'polynomial',
    'illumination_degree': 4,
    'illumination_dark': 0.0,
    'illumination_per_plate': True,
    'illumination_max_fields': 50,
    'illumination_qc': True,
    'illumination_on_missing': 'error',

    # Measurement Features
    # Optional settings
    'save_measurements': True,
    'calculate_correlation': True,
    'corrected_manders': False,
    'spatial_measurements': False,
    'spatial_neighbor_radius': 50,
    'manders_thresholds': [15, 85, 95],
    'homogeneity': True,
    'homogeneity_distances': [8, 16, 32],
    'radial_dist': True,
    'distance_gaussian_sigma': 10,
    'object_distances': False,
    'object_distance_maxima': True,
    'object_distance_intensity': True,

    # Object Filtering
    # Optional settings
    'uninfected': True,
    'cell_min_size': 0,
    'cell_max_size': None,
    'cytoplasm_min_size': 0,
    'nucleus_min_size': 0,
    'nucleus_max_size': None,
    'pathogen_min_size': 0,
    'pathogen_max_size': None,
    'merge_edge_pathogen_cells': True,

    # Crop Output
    # Required settings
    'crop_mode': ['cell'],
    # Conditionally required settings
    'normalize': False,
    # Optional settings
    'save_png': True,
    'save_arrays': False,
    'png_size': [224, 224],
    'png_channel_mapping': {'r': 2, 'g': 1, 'b': 0},
    'dialate_pngs': False,
    'dialate_png_ratios': [0.2],
    'use_bounding_box': False,
    'normalize_by': 'png',

    # Preview & Diagnostics
    # Optional settings
    'plot': False,
    'test_mode': False,
    'test_nr': 10,

    # 3D Calibration (Beta)
    # Optional settings
    'anisotropy': None,
    'voxel_size_z_um': None,
    'voxel_size_xy_um': None,

    # Runtime & Reliability
    # Optional settings
    'resume': False,
    'strict_errors': None,
    'max_failure_rate': None,
    'dry_run': False,
    'verbose': False,
    'n_jobs': max(1, (__import__('os').cpu_count() or 1) - 2),
}

In [ ]:
settings.update({
    # Mask & Channel Mapping
    # Optional settings
    'number_of_organelles': 0,
    'organelle_mask_dim': None,
    'organelleb_mask_dim': None,
    'organellec_mask_dim': None,
    'organelled_mask_dim': None,
    'organelle_type': 'custom',
    'organelleb_type': 'custom',
    'organellec_type': 'custom',
    'organelled_type': 'custom',

    # Measurement Features
    # Optional settings
    'summarize_organelles_by': 'cell',

    # Object Filtering
    # Optional settings
    'organelle_min_size': 0,
    'organelleb_min_size': 0,
    'organellec_min_size': 0,
    'organelled_min_size': 0,
})

In [ ]:
measure_crop(settings)

## Outputs and next steps

`measurements/measurements.db` and, when enabled, object crops under `data/`.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)